In [16]:
import json
import os
import soundfile as sf
import jsonlines
from pathlib import Path

start_lyrics = "[beginning]\n\n"
middle_lyrics = "[middle]\n\n"
end_lyrics = "[end]\n\n"
output_jsonl_path = "output.jsonl"
audio_dir_path = "C:/Users/adaml/Documents/YuE_finetune_trans_gen/test_music"
codes_dir_path = "C:/Users/adaml/Documents/YuE_finetune_trans_gen/test_music_codes"
json_obj = {}
#since bumch of audio files in FMA_large are corrupted, we cannot use the original dataset to get all the track names.
track_names = list(set([file.stem for file in Path(audio_dir_path).rglob('*.mp3') if file.is_file()]))

with open(output_path, 'w') as file:
    pass

for current_id, track_name in enumerate(track_names):
    #in the example, id starts at 1
    json_obj["id"] = str(current_id + 1)
    
    #The vocals and instrumental stems for the same track should have the same duration
    mixture_audio_path = os.path.join(audio_dir_path, track_name + ".mp3")
    

    vocals_codes_path = os.path.join(codes_dir_path, track_name + ".Vocals.npy")
    instrumental_codes_path = os.path.join(codes_dir_path, track_name + ".Instrumental.npy")
    mixture_codes_path = os.path.join(codes_dir_path, track_name + ".npy")
    
#     if (not (os.path.exists(vocals_codes_path) 
#              and os.path.exists(instrumental_codes_path) 
#              and os.path.exists(mixture_codes_path))):
#         print(f"Missing codec files for track {track_name}!")
    
    json_obj["codec"] = mixture_codes_path
    json_obj["vocals_codec"] = vocals_codes_path
    json_obj["instrumental_codec"] = instrumental_codes_path
    
    #get track duration in seconds, so that we know the split time for start, middle and end
    track_info = sf.info(mixture_audio_path)
    track_duration = round(track_info.frames / track_info.samplerate, 2)
    
    codec_fps = 50
    segment_duration = round(track_duration / 3, 2)
    segment_codes_duration = int(segment_duration * codec_fps)
    
    msa_start = {"start": 0.0, "end": segment_duration, "label": "beginning"}
    
    msa_middle = {"start": segment_duration, "end": segment_duration * 2.0, "label": "middle"}
    
    msa_end = {"start": segment_duration * 2.0, "end": track_duration, "label": "end"}
    
    json_obj["audio_length_in_sec"] = round(track_duration, 2)
    
    json_obj["msa"] = [msa_start, msa_middle, msa_end]
    
    json_obj["genres"] = "" #need the metadata csv file
    
    
    
    start_lyric_segment = {"offset": 0.0, 
                           "duration": segment_duration, 
                           "label": "beginning", 
                           "codec_frame_start":0, 
                           "codec_frame_end": segment_codes_duration,
                           "line_content": start_lyrics}
    
    middle_lyric_segment = {"offset": segment_duration, 
                            "duration": segment_duration,
                            "label": "middle",
                            "codec_frame_start": segment_codes_duration,
                            "codec_frame_end": segment_codes_duration * 2,
                            "line_content": middle_lyrics}
    
    end_lyric_segment = {"offset": segment_duration * 2,
                         "duration": segment_duration,
                         "label": "end",
                         "codec_frame_start": segment_codes_duration * 2,
                         "codec_frame_end": int(track_duration * codec_fps)}
    
    segmented_lyrics =  {"segmented_lyrics": [start_lyric_segment, middle_lyric_segment, end_lyric_segment]}
    
    json_obj["splitted_lyrics"] = segmented_lyrics
    
    with open(output_jsonl_path, "a") as f:
        json.dump(json_obj, f, separators=(",", ":"))
        f.write("\n")


In [2]:
pip install jsonlines

Note: you may need to restart the kernel to use updated packages.
